In [1]:
import warnings
warnings.filterwarnings('ignore')

In [17]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process
import ftfy

from zipfile import ZipFile
from urllib.request import urlopen
from io import BytesIO

In [7]:
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQZC7cVru6ltLR2e8XN5jdJPfxfj42BAxUApe3Zq3_ENQjLtYntmAxD0pIHqEUJ4ZFLXlybKJdkLf2r/pub?output=csv'
candinfo = pd.read_csv(url)
candinfo.to_csv('../../2026_data/2026_midterms_candidateinfo.csv')
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any
0,AK-AL,NaN,Nick Begich,False,True
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False
2,AL-02,Shomari Figures,Hampton Harris,True,False
3,AL-03,Lee McInnis,Mike Rogers,False,True
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True


In [8]:
for party in ['dem', 'rep']:
    candinfo[f'{party}_cand'] = candinfo[f'{party}_cand'].fillna('TBD').astype(str)
    candinfo[f'{party}_inc_any'] = candinfo[f'{party}_inc_any'].astype(bool)

In [9]:
dem_only_candinfo = candinfo[candinfo['rep_cand'] == 'Not Contested']
rep_only_candinfo = candinfo[candinfo['dem_cand'] == 'Not Contested']
candinfo = candinfo[(candinfo['dem_cand'] != 'Not Contested') &
    (candinfo['rep_cand'] != 'Not Contested')]

In [38]:
candinfo['state_po'] = candinfo['cd'].map(lambda x: x[:2]).astype(str)
candinfo['district_number'] = candinfo['cd'].map(lambda x: 0 if x[3:] == 'AL' else int(x[3:]))
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number
0,AK-AL,TBD,Nick Begich,False,True,AK,0
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4


In [39]:
candinfo.shape

(423, 7)

In [18]:
# Code snippet courtesy of hantoine: https://gist.github.com/hantoine/c4fc70b32c2d163f604a8dc2a050d5f6
def download_and_unzip(url, extract_to='.'):
    http_candinfoponse = urlopen(url)
    zipfile = ZipFile(BytesIO(http_candinfoponse.read()))
    zipfile.extractall(path=extract_to)

In [28]:
fec_webl_url = 'https://www.fec.gov/files/bulk-downloads/2026/webl26.zip'
fec_cn_url = 'https://www.fec.gov/files/bulk-downloads/2026/cn26.zip'
download_and_unzip(fec_webl_url, extract_to='../../2026_data/fec')
download_and_unzip(fec_cn_url, extract_to='../../2026_data/fec')

In [20]:
fec_webl_colnames = ["CAND_ID", "CAND_NAME", "CAND_ICI", "PTY_CD", "CAND_PTY_AFFILIATION", "TTL_RECEIPTS", "TRANS_FROM_AUTH", "TTL_DISB", "TRANS_TO_AUTH", "COH_BOP", "COH_COP", "CAND_CONTRIB", "CAND_LOANS", "OTHER_LOANS", "CAND_LOAN_REPAY", "OTHER_LOAN_REPAY", "DEBTS_OWED_BY", "TTL_INDIV_CONTRIB", "CAND_OFFICE_ST", "CAND_OFFICE_DISTRICT", "SPEC_ELECTION", "PRIM_ELECTION", "RUN_ELECTION", "GEN_ELECTION", "GEN_ELECTION_PRECENT", "OTHER_POL_CMTE_CONTRIB", "POL_PTY_CONTRIB", "CVG_END_DT", "INDIV_REFUNDS", "CMTE_REFUNDS"]

In [29]:
fec_cn_colnames = ["CAND_ID", "CAND_NAME", "CAND_PTY_AFFILIATION", "CAND_ELECTION_YR", "CAND_OFFICE_ST", "CAND_OFFICE", "CAND_OFFICE_DISTRICT", "CAND_ICI", "CAND_STATUS", "CAND_PCC", "CAND_ST1", "CAND_ST2", "CAND_CITY", "CAND_ST", "CAND_ZIP"]

In [33]:
webl = pd.read_table('../../2026_data/fec/webl26.txt', sep="|", names=fec_webl_colnames)
cn = pd.read_table('../../2026_data/fec/cn.txt', sep="|", names=fec_cn_colnames)
fec = pd.merge(left=webl, right=cn, on='CAND_ID', how='left')
fec = fec[[col for col in fec.columns.values if ('_y' not in col)]]
fec.columns = fec.columns.str.strip('_x')
fec.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CMTE_REFUNDS,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H2AK01158,"PELTOLA, MARY",C,1,DEM,152304.86,0.00,232791.29,0.0,83969.49,...,0.00,2026.0,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501.0
1,H6AK01084,"SCHULTZ, MATTHEW DAMIAN",C,1,DEM,579656.42,0.00,231366.34,1075.0,0.00,...,0.00,2026.0,H,C,C00923714,PO BOX 240641,NaN,ANCHORAGE,AK,99524.0
2,H2AK01083,"BEGICH, NICHOLAS III",I,2,REP,4307322.69,1302998.38,1581037.44,0.0,104330.06,...,3382.86,2026.0,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567.0
3,H6AK01092,"HILL, BILL",C,3,IND,783044.48,0.00,187786.59,0.0,0.00,...,0.00,2026.0,H,C,C00935437,PO BOX 220703,NaN,ANCHORAGE,AK,99522.0
4,H6AL01094,"JONES, CLYDE W MR. JR",O,1,DEM,37487.19,0.00,18859.98,0.0,0.00,...,0.00,2026.0,H,C,C00920918,11637 WENTWOOD CT,NaN,DAPHNE,AL,36526.0


In [34]:
fec.columns.values

array(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'CAND_ELECTION_YR', 'CAND_OFFICE',
       'CAND_STATUS', 'CAND_PCC', 'CAND_ST1', 'CAND_ST2', 'CAND_CITY',
       'CAND_ST', 'CAND_ZIP'], dtype=object)

In [36]:
fec = fec[fec['CAND_OFFICE'] == 'H']

In [37]:
fec.shape

(2334, 39)

In [40]:
def get_fuzzymatch_cand(state_po, district, candidate):
    df = fec[(fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_OFFICE_DISTRICT'] == district)]

    if '/' in candidate:
        cands = candidate.split(separator='/')
        matches = []
        for c in cands:
            fuzzymatch = process.extractOne(c, df['CAND_NAME'].values, scorer=fuzz.WRatio, score_cutoff=55)
            if fuzzymatch is not None:
                matches.append(fuzzymatch[0])
            else:
                matches.append('NaN')
        return matches

    else: 
        fuzzymatch = process.extractOne(candidate, df['CAND_NAME'].values, scorer=fuzz.WRatio, score_cutoff=55)
        return fuzzymatch if fuzzymatch is None else fuzzymatch[0]